In [1]:
import bw2data, bw2io, bw2calc
import numpy as np
import os
import re
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)

In [2]:
wind  = pd.read_excel("../dp-LCI_output/staticLCI_(p)GWP100/wind_CN_US_staticLCI_threeGWP100.xlsx")
hydro = pd.read_excel("../dp-LCI_output/staticLCI_(p)GWP100/hydro_reservoir_CN-asRoW_US-asQC_staticLCI_threeGWP100_wo_updatingCO2CH4.xlsx")
pv    = pd.read_excel("../dp-LCI_output/staticLCI_(p)GWP100/PV_CN_US_staticLCI_threeGWP100.xlsx")

In [3]:
def add_diff_columns(df):
    df = df.copy()

    df["diff%_pGWP-fixedCO2"] = (
        (df["pGWP100_fixedCO2"] - df["gwp100"]) / df["gwp100"] * 100
    ).round(0).astype(int)

    df["diff%_pGWP-dpCO2"] = (
        (df["pGWP100_dpCO2"] - df["gwp100"]) / df["gwp100"] * 100
    ).round(0).astype(int)

    return df


In [4]:
hydro = add_diff_columns(hydro)
wind  = add_diff_columns(wind)
pv    = add_diff_columns(pv)

In [5]:
def highlight_threshold(val):
    if pd.isna(val):
        return ""
    if abs(val) >= 10:
        return "background-color: #ff6b6b; color: black;"   # red
    elif abs(val) >= 5:
        return "background-color: #f4b084; color: black;"   # yellow
    else:
        return ""



In [6]:

hydro.style.applymap(
        highlight_threshold,
        subset=["diff%_pGWP-fixedCO2", "diff%_pGWP-dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )




/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_2243/4045844554.py:1: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  hydro.style.applymap(


,Activity,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2,diff%_pGWP-fixedCO2,diff%_pGWP-dpCO2
0,"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2030",0.015721,0.017909,0.015650,14,0
1,"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2040",0.015028,0.016988,0.015003,13,0
2,"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2050",0.013658,0.015535,0.013646,14,0
3,"market for electricity, hydro, high voltage, US, SSP2-M, 2030",0.015746,0.017505,0.015649,11,-1
4,"market for electricity, hydro, high voltage, US, SSP2-M, 2040",0.015562,0.016343,0.015522,5,0
5,"market for electricity, hydro, high voltage, US, SSP2-M, 2050",0.015461,0.015388,0.015462,0,0
6,"market for electricity, hydro, high voltage, US, SSP5-H, 2030",0.015801,0.017236,0.015666,9,-1
7,"market for electricity, hydro, high voltage, US, SSP5-H, 2040",0.015678,0.015613,0.015559,0,-1
8,"market for electricity, hydro, high voltage, US, SSP5-H, 2050",0.015582,0.014036,0.015528,-10,0
9,"market for electricity, hydro, high voltage, CN, SSP1-VLLO, 2030",0.050523,0.057266,0.050021,13,-1


In [7]:
# wind and PV has 2040 rows, and we don't use them 
wind = wind[wind["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]

wind.style.applymap(
        highlight_threshold,
        subset=["diff%_pGWP-fixedCO2", "diff%_pGWP-dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )


/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_2243/2567930173.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  wind = wind[wind["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]
/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_2243/2567930173.py:4: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  wind.style.applymap(


,Activity,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2,diff%_pGWP-fixedCO2,diff%_pGWP-dpCO2
0,"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030",0.036905,0.041520,0.036365,13,-1
2,"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050",0.017832,0.019952,0.017591,12,-1
3,"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2030",0.037723,0.041415,0.037095,10,-2
5,"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050",0.031185,0.030810,0.030951,-1,-1
6,"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030",0.038734,0.041703,0.037964,8,-2
8,"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2050",0.034395,0.030785,0.033988,-10,-1
9,"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2030",0.020637,0.023218,0.020336,13,-1
11,"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2050",0.009972,0.011157,0.009837,12,-1
12,"market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030",0.021095,0.023160,0.020744,10,-2
14,"market for electricity, wind, high voltage, US-TRE, SSP2-M, 2050",0.017439,0.017229,0.017308,-1,-1


In [8]:
pv = pv[pv["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]

pv.style.applymap(
        highlight_threshold,
        subset=["diff%_pGWP-fixedCO2", "diff%_pGWP-dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )


/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_2243/3754451970.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pv = pv[pv["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]
/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_2243/3754451970.py:3: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  pv.style.applymap(


,Activity,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2,diff%_pGWP-fixedCO2,diff%_pGWP-dpCO2
0,"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030",0.032003,0.035375,0.030962,11,-3
2,"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2050",0.010065,0.010716,0.009459,6,-6
3,"market for electricity, PV, low voltage, CN, SSP2-M, 2030",0.032911,0.035515,0.031793,8,-3
5,"market for electricity, PV, low voltage, CN, SSP2-M, 2050",0.020241,0.019521,0.019611,-4,-3
6,"market for electricity, PV, low voltage, CN, SSP5-H, 2030",0.034533,0.036574,0.033279,6,-4
8,"market for electricity, PV, low voltage, CN, SSP5-H, 2050",0.024782,0.021741,0.024011,-12,-3
9,"market for electricity, PV, low voltage, US, SSP1-VLLO, 2030",0.020103,0.022236,0.019463,11,-3
11,"market for electricity, PV, low voltage, US, SSP1-VLLO, 2050",0.006560,0.007007,0.006184,7,-6
12,"market for electricity, PV, low voltage, US, SSP2-M, 2030",0.020738,0.022395,0.020048,8,-3
14,"market for electricity, PV, low voltage, US, SSP2-M, 2050",0.013044,0.012596,0.012654,-3,-3
